In [31]:
!pip install emoji

In [32]:
import json
import re
import random
from collections import Counter
import emoji

# **Loan Jsonl File**

In [33]:
INPUT_FILE  = "commit_dataset.jsonl"
OUTPUT_FILE = "commit_dataset_clean.jsonl"

In [34]:
def load_jsonl(filepath):

    examples = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue  # skip blank lines
            try:
                examples.append(json.loads(line))
            except json.JSONDecodeError:
                print(f"Warning: Line {line_number} was not valid JSON, skipping.")
    return examples

print("Loading raw dataset...")
raw_data = load_jsonl(INPUT_FILE)
print(f"Loaded {len(raw_data)} raw examples")


Loading raw dataset...
Loaded 1000 raw examples


# **Cleaning Functions**

In [ ]:
# Clean up a raw git diff.

def clean_diff(diff_text):

    if not diff_text:
        return None

    lines = diff_text.split("\n")
    cleaned_lines = []

    for line in lines:
        # Skip git metadata lines (not actual code)
        if line.startswith("index "):         continue  # git object hashes
        if line.startswith("old mode"):       continue  # file permissions
        if line.startswith("new mode"):       continue  # file permissions
        if line.startswith("Binary files"):   continue  # binary, not useful
        if line.startswith("similarity index"): continue
        if line.startswith("rename from"):    continue
        if line.startswith("rename to"):      continue

        # Skip extremely long lines (usually minified JS or data blobs)
        if len(line) > 300:
            continue

        cleaned_lines.append(line)

    result = "\n".join(cleaned_lines).strip()

    # Return None if nothing useful remains
    if len(result) < 20:
        return None

    return result


In [36]:
def remove_emojis(text):

    # Remove unicode emoji characters (🐛 ✨ 🔥 etc.)

    text = emoji.replace_emoji(text, replace="")

    # Remove :shortcode: style emojis (:bug: :sparkles: etc.)

    text = re.sub(r":[a-z0-9_+-]+:", "", text)

    # Clean up any double spaces left behind

    text = re.sub(r"\s{2,}", " ", text).strip()

    return text

In [ ]:
# Clean commit message and exclude github auto generated, version bump, jira ticket text etc.

def clean_commit_message(message):

    if not message:
        return None

    # Take only the first line
    first_line = message.strip().split("\n")[0].strip()
    first_line = remove_emojis(first_line)

    # Remove issue tracker tags at the start: [JIRA-123], (#456), etc.
    first_line = re.sub(r"^\[.*?\]\s*", "", first_line)
    first_line = re.sub(r"^\(#\d+\)\s*", "", first_line)
    first_line = re.sub(r"\s*\(#\d+\)+\s*$", "", first_line).strip()
    first_line = re.sub(r"\s*#\d+", "", first_line).strip()
    first_line = first_line.strip()



    # Too short
    if len(first_line) < 10:
        return None

    # Too long
    if len(first_line) > 100:
        return None

    # Version bumps — boring and repetitive
    if re.match(r"^(bump|update|release|version)\s+v?\d+\.", first_line, re.IGNORECASE):
        return None

    # Merge commits — not useful patterns
    if re.match(r"^merge\b", first_line, re.IGNORECASE):
        return None

    # Automated dependency updates (Dependabot etc.)
    if "bump" in first_line.lower() and re.search(r"\d+\.\d+", first_line):
        return None

    if first_line == first_line.upper() and len(first_line) > 5:
        return None

    # Starts with a number — usually a ticket ID, not a real message
    if re.match(r"^\d+", first_line):
        return None

    return first_line


In [38]:
# Check if the diff actually shows meaningful code changes.

def is_useful_diff(diff_text):

    if not diff_text:
        return False

    lines = diff_text.split("\n")

    # Count actual code change lines
    added_lines   = sum(1 for l in lines if l.startswith("+") and not l.startswith("+++"))
    removed_lines = sum(1 for l in lines if l.startswith("-") and not l.startswith("---"))

    # Need at least 1 added or removed line
    if added_lines + removed_lines < 1:
        return False

    # Skip diffs that are ONLY adding/removing blank lines (whitespace fixes)
    meaningful = [
        l for l in lines
        if (l.startswith("+") or l.startswith("-"))
        and not l.startswith("+++")
        and not l.startswith("---")
        and l.strip() not in ("+", "-", "")
    ]
    if len(meaningful) < 1:
        return False

    return True



In [39]:
# Wrap the diff and commit message in the Alpaca instruction format.

def format_for_training(diff_text, commit_message):

    # Truncate diff to 1500 chars max — keeps context window manageable
    if len(diff_text) > 1500:
        diff_text = diff_text[:1500] + "\n... [truncated]"

    text = (
        "### Instruction:\n"
        "You are an expert developer. "
        "Given the git diff below, write a concise and descriptive "
        "commit message that explains what changed and why.\n\n"
        f"Git diff:\n```\n{diff_text}\n```\n\n"
        "### Response:\n"
        f"{commit_message}"
    )

    return text



In [40]:
def clean_dataset(raw_data):

    clean_examples = []
    skip_reasons   = Counter()

    for i, example in enumerate(raw_data):

        raw_diff    = example.get("instruction", "")
        raw_message = example.get("response", "")

        # Extract the actual diff text from the instruction string
        diff_match = re.search(r"```\n(.*?)```", raw_diff, re.DOTALL)
        diff_text  = diff_match.group(1) if diff_match else raw_diff

        clean_diff_text = clean_diff(diff_text)
        clean_message   = clean_commit_message(raw_message)

        if not clean_diff_text:
            skip_reasons["bad diff"] += 1
            continue

        if not clean_message:
            skip_reasons["bad message"] += 1
            continue

        if not is_useful_diff(clean_diff_text):
            skip_reasons["no real changes"] += 1
            continue

        formatted_text = format_for_training(clean_diff_text, clean_message)

        clean_examples.append({
            "text": formatted_text,
            "commit_message": clean_message,
            "repo": example.get("repo", ""),
        })

    return clean_examples, skip_reasons


print("\nCleaning dataset...")
clean_data, skip_reasons = clean_dataset(raw_data)

print(f"\nResults:")
print(f"  Started with : {len(raw_data)} examples")
print(f"  Kept         : {len(clean_data)} examples")
print(f"  Removed      : {len(raw_data) - len(clean_data)} examples")
print(f"\nWhy examples were removed:")
for reason, count in skip_reasons.most_common():
    print(f"  {reason:20s} : {count}")



Cleaning dataset...

Results:
  Started with : 1000 examples
  Kept         : 957 examples
  Removed      : 43 examples

Why examples were removed:
  bad message          : 23
  no real changes      : 20


In [ ]:
# Remove examples with identical commit messages. We keep the first occurrence of each unique message.
def deduplicate(examples):
  
    seen_messages = set()
    unique = []

    for ex in examples:
        msg = ex["commit_message"].lower().strip()
        if msg not in seen_messages:
            seen_messages.add(msg)
            unique.append(ex)

    return unique

before_dedup = len(clean_data)
clean_data = deduplicate(clean_data)
print(f"\nAfter deduplication: {len(clean_data)} examples (removed {before_dedup - len(clean_data)} duplicates)")


After deduplication: 746 examples (removed 211 duplicates)


In [42]:
random.seed(42)  # makes the shuffle reproducible
random.shuffle(clean_data)

split_point  = int(len(clean_data) * 0.9)
train_data   = clean_data[:split_point]
val_data     = clean_data[split_point:]

print(f"\nData split:")
print(f"  Training   : {len(train_data)} examples")
print(f"  Validation : {len(val_data)} examples")



Data split:
  Training   : 671 examples
  Validation : 75 examples


In [ ]:
def save_jsonl(data, filepath):
    with open(filepath, "w", encoding="utf-8") as f:
        for example in data:
            f.write(json.dumps(example, ensure_ascii=False) + "\n")
    print(f"  Saved {len(data)} examples → {filepath}")

print("\nSaving files...")
save_jsonl(train_data, "train.jsonl")
save_jsonl(val_data,   "val.jsonl")
save_jsonl(clean_data, OUTPUT_FILE)  



In [44]:
def inspect_examples(examples, n=3):
    print(f"\n{'='*60}")
    print(f"  SAMPLE TRAINING EXAMPLES (showing {n})")
    print(f"{'='*60}")

    samples = random.sample(examples, min(n, len(examples)))

    for i, ex in enumerate(samples, 1):
        print(f"\n── Example {i} ──────────────────────────────────")
        print(f"Repo   : {ex['repo']}")
        print(f"Target : {ex['commit_message']}")
        print(f"\nFull training text:")
        print(ex["text"])
        print()

inspect_examples(train_data)


  SAMPLE TRAINING EXAMPLES (showing 3)

── Example 1 ──────────────────────────────────
Repo   : encode/httpx
Target : Docs: Add `httpx.Proxy` to api.md

Full training text:
### Instruction:
You are an expert developer. Given the git diff below, write a concise and descriptive commit message that explains what changed and why.

Git diff:
```
diff --git a/docs/api.md b/docs/api.md
--- a/docs/api.md
+++ b/docs/api.md
@@ -159,3 +159,18 @@ what gets sent over the wire.*
 * `def delete(name, [domain], [path])`
 * `def clear([domain], [path])`
 * *Standard mutable mapping interface*
+
+## `Proxy`
+
+*A configuration of the proxy server.*
+
+
```

### Response:
Docs: Add `httpx.Proxy` to api.md


── Example 2 ──────────────────────────────────
Repo   : psf/requests
Target : Update Sphinx to work with latest readthedocs requirements

Full training text:
### Instruction:
You are an expert developer. Given the git diff below, write a concise and descriptive commit message that explains what cha

In [ ]:
def final_summary(train_data, val_data):
    all_data = train_data + val_data

    prefixes = Counter()
    for ex in all_data:
        msg = ex["commit_message"].lower()
        if msg.startswith("fix"):      prefixes["fix"] += 1
        elif msg.startswith("feat"):   prefixes["feat"] += 1
        elif msg.startswith("add"):    prefixes["add"] += 1
        elif msg.startswith("update"): prefixes["update"] += 1
        elif msg.startswith("refactor"): prefixes["refactor"] += 1
        elif msg.startswith("remove"): prefixes["remove"] += 1
        else:                          prefixes["other"] += 1

    avg_len = sum(len(ex["text"]) for ex in all_data) / len(all_data)

    print(f"\n{'='*60}")
    print(f"  FINAL DATASET SUMMARY")
    print(f"{'='*60}")
    print(f"  Total clean examples : {len(all_data)}")
    print(f"  Training set         : {len(train_data)}")
    print(f"  Validation set       : {len(val_data)}")
    print(f"  Avg text length      : {avg_len:.0f} chars")
    print(f"\n  Commit message types:")
    for prefix, count in prefixes.most_common():
        bar = "█" * (count * 20 // len(all_data))
        print(f"    {prefix:12s} {count:4d}  {bar}")
    print(f"{'='*60}")

final_summary(train_data, val_data)
